<a href="https://colab.research.google.com/github/gmauricio-toledo/tda-gdl/blob/main/T02-PCA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1>Tarea 2</h1>

**Fecha de entrega:** Martes 30 de Septiembre, por Teams


![Pipeline del Machine Learning](https://drive.google.com/uc?id=1S9KVyZbkiciIEeC7cLi-epa62gfFM0pi)



# Ejercicio 1: Distorsión de las distancias con PCA

1. Genera un dataset sintético de 1000 puntos en 300 dimensiones. Puedes usar las funciones [`make_blobs`](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_blobs.html), [`make_classification`](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_classification.html), ...
2. Calcula la matriz de distancias euclidianas entre todos los pares de puntos. Puedes usar la función [`pairwise_distances`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise_distances.html)
3. Aplica PCA al dataset y proyecta a 2D.
4. Calcula la matriz de distancias euclidianas en el espacio 2D con la misma función.
5. Grafica con matplotlib:
 * Un `scatterplot` de la distancia original (eje X) vs. la distancia proyectada con PCA (eje Y). Deberías ver que todos los puntos están debajo de la línea $y=x$, esto da evidencia de la propiedad
  $$\text{dist_proy} \leq \text{dist_original}$$
 * Un histograma del ratio $\displaystyle \frac{\text{dist_proy}}{\text{dist_original}}$. Verás que este ratio $r$ siempre satisface $r\leq 1$

# Ejercicio 2: PCA en una superficie cuadrática

* Samplea 500 puntos sobre la superficie de un toro o un paraboloide hiperbólico en $\mathbb{R}^3$.
* Agrega ruido normal a los puntos en la coordenada $z$.
* Grafica los puntos en $\mathbb{R}^3$, puedes usar la función `scatter_plot_3d_plotly` de la notebook de [PCA](https://github.com/gmauricio-toledo/tda-gdl/blob/main/06-PCA.ipynb).
* Haz PCA 2d a los puntos y graficalos.
* Compara ambas gráficas y observa qué tan adecuado fué usar PCA en este caso.

# Ejercicio 3

En esta práctica, trabajaremos con un conjunto de datos que registra actividades humanas mediante sensores de un dispositivo móvil. El estudio original se realizó con 30 voluntarios de entre 19 y 48 años, quienes llevaron un smartphone en la cintura mientras realizaban seis actividades cotidianas: caminar, subir escaleras, bajar escaleras, estar sentado, estar de pie y estar acostado. Los sensores del dispositivo capturaron datos de aceleración y velocidad angular, y las actividades fueron grabadas en video para etiquetar los datos de manera precisa.
El conjunto de datos se dividió en dos partes: 70% para entrenamiento y 30% para prueba. Cada registro incluye información de los sensores, así como la etiqueta de la actividad realizada y el identificador del voluntario.
El objetivo principal de esta práctica es explorar el efecto de la reducción de dimensionalidad con PCA en el rendimiento de modelos de clasificación, utilizando las características extraídas de los datos de los sensores. Este ejercicio nos permitirá entender cómo técnicas como PCA pueden mejorar la eficiencia y precisión en problemas de aprendizaje automático.

[Fuente del dataset](https://archive.ics.uci.edu/dataset/240/human+activity+recognition+using+smartphones)

## Instrucciones

1. Imprimir la forma del dataset **indicando el número de puntos y la dimensión de los datos**.
2. ¿Cuántas clases tenemos en este problema? Para esto, explora el vector de clases `y_train`, `y_test`.
3. Usar alguna técnica de preprocesamiento que consideres adecuada basándote en las variables del problema. **Cuidado con el data leakage:** Entrena con el conjunto de entrenamiento y transforma (sin entrenar) el conjunto de prueba.
4. Entrena un clasificador `sklearn.svm.SVC` para este problema de clasificación. Reporta las métricas accuracy en el conjunto de entrenamiento y prueba.
5. Realiza la reducción de dimensionalidad PCA, decide el número de componentes basándote en la varianza explicada y/o el accuracy obtenido en la tarea de clasificación del punto siguiente.
6. Entrena un clasificador `sklearn.svm.SVC` para el conjunto de datos transformado con PCA. Reporta las métricas accuracy en el conjunto de entrenamiento y prueba.
7. Grafica una reducción de dimensionalidad 2D con el conjunto de datos de entrenamiento, colorea los puntos de acuerdo a la clase.

Bajar y descomprimir el conjunto de datos

In [ ]:
!gdown 1TYKt5VjduXHaeG8Q9uZj1YJ4mbuzC3cJ  # Bajar una copia desde google drive
!unrar x UCI_HAR_Dataset.rar  # Descomprimir

Leer el conjunto de datos

In [ ]:
import pandas as pd
import numpy as np

# Leemos los dataframes
x_train_df = pd.read_csv('/content/UCI HAR Dataset/UCI HAR Dataset/train/X_train.txt',
                         sep=r'\s+',
                         header=None)
x_test_df = pd.read_csv('/content/UCI HAR Dataset/UCI HAR Dataset/test/X_test.txt',
                        sep=r'\s+',
                        header=None)

# Los convertimos a arreglos de numpy
X_train = x_train_df.values
X_test = x_test_df.values

# Leemos los arreglos de numpy con las etiquetas de clases
y_train = np.loadtxt('/content/UCI HAR Dataset/UCI HAR Dataset/train/y_train.txt')
y_test = np.loadtxt('/content/UCI HAR Dataset/UCI HAR Dataset/test/y_test.txt')

Información de las features, por si quieren saber más:

In [ ]:
features_names_df = pd.read_csv('/content/UCI HAR Dataset/UCI HAR Dataset/features.txt',
                                sep=r'\s+',
                                header=None,
                                names=['Feature','Meaning'])
features_names_df

In [ ]:
labels_names = {1: 'WALKING',
                2: 'WALKING_UPSTAIRS',
                3: 'WALKING_DOWNSTAIRS',
                4: 'SITTING',
                5: 'STANDING',
                6: 'LAYING'}

Observa la distribución de clases

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axs = plt.subplots(1,2, figsize=(8,4))
sns.countplot(x=y_train, ax=axs[0])
axs[0].set_xticks(range(len(labels_names)))
axs[0].set_xticklabels(labels_names.values(),rotation=90)
axs[0].set_title('Train')
sns.countplot(x=y_test, ax=axs[1])
axs[1].set_xticks(range(len(labels_names)))
axs[1].set_xticklabels(labels_names.values(),rotation=90)
axs[1].set_title('Test')
fig.show()